# Recommendation System with ALS (Day 9)

## Objective
Learn **Collaborative Filtering** using **Alternating Least Squares (ALS)** to build recommendation systems.

## Challenge Context
The official challenge uses **e-commerce data** (users, products, ratings) to recommend products. 

**Our Adaptation for Energy Data**:  
We'll build a **time period recommendation system** that:
* **Recommends similar time periods** based on energy patterns
* **Identifies optimal production windows** for H2 electrolysis
* **Predicts favorable conditions** for energy-intensive operations

## What is ALS?

**Alternating Least Squares (ALS)** is a matrix factorization technique for collaborative filtering:

```
User-Item Matrix (sparse) → User Factors × Item Factors (dense)

Example:
        Item1  Item2  Item3
User1    5      ?      3
User2    ?      4      ?
User3    2      5      4

↓ ALS Factorization ↓

User Factors (latent features) × Item Factors (latent features)
= Predicted ratings for missing values
```

## Honest Assessment: ALS Fit for This Problem

> **ALS is included here to fulfill the Day 9 challenge requirement.** In this current form, it is an artificial adaptation — not the ideal algorithm for energy production scheduling.

**Why ALS is a stretch here:**

| ALS Requirement | Our Reality |
| --- | --- |
| Many users with diverse preferences | Only 5 synthetic strategies |
| Real user-item interactions | Heuristic ratings we compute ourselves |
| Sparse matrix to fill | We already know all conditions |
| Discover hidden patterns | A SQL `WHERE price < 30 ORDER BY price` works better |

**When ALS WOULD genuinely help (future scope):**
* **Real operators** across multiple countries making actual production decisions
* **Cross-country patterns**: "Operators in DE who produced during wind peaks also benefited from similar windows in NL"
* **Many strategies/users** (50+ plant operators) with genuinely different, unknown preferences
* **Implicit feedback** from actual production logs (ran electrolyzer or not)

## Our Energy Adaptation

**User-Item Mapping**:  
* **"Users"** = Operating Strategies (e.g., high_load, low_price, renewable_peak)
* **"Items"** = Time Periods (hourly records)
* **"Ratings"** = Suitability Score (based on conditions)

## Challenge Tasks (Day 9)
✅ Create rating mapping  
✅ Train ALS model  
✅ Generate Top-5 recommendations

In [0]:
from pyspark.ml.recommendation import ALS
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd

# Configuration
CATALOG = "dbacademy"
SCHEMA = "labuser13955035_1772876383"
BASE_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_model_scoring_base"

print("🎯 Recommendation System for Energy Data")
print("="*80)
print(f"Source Table: {BASE_TABLE}")
print(f"Task: Recommend optimal time periods for H2 production")
print("="*80)

In [0]:
# Load energy data
df = spark.table(BASE_TABLE)

print("\n📊 Dataset Overview:")
print(f"Total records: {df.count():,}")
print(f"Time range: {df.agg(F.min('event_time_utc'), F.max('event_time_utc')).collect()[0]}")

print("\n🔑 Key Features for Recommendations:")
display(df.select(
    "event_time_utc",
    "price_eur_mwh",
    "avg_actual_total_load_mw",
    "renewable_share_pct",
    "temperature_c",
    "wind_speed_ms"
).limit(5))

## Energy Operating Strategies

We'll define **5 operating strategies** that represent different H2 production priorities:

| Strategy | Goal | Preferred Conditions |
| --- | --- | --- |
| **COST_OPTIMIZER** | Minimize electricity cost | Low price, any renewable |
| **GREEN_PRIORITY** | Maximize renewable usage | High renewable %, any price |
| **LOAD_BALANCER** | Operate during low demand | Low grid load, any price |
| **OPPORTUNISTIC** | Best overall conditions | Low price + high renewable |
| **BASELOAD** | Steady 24/7 operation | Moderate everything |

Each strategy will have different preferences for time periods.

In [0]:
print("✅ STEP 1: Create rating mapping (suitability scores)")
print("="*80)

# Create unique time period IDs
df_with_id = df.withColumn(
    "time_period_id",
    F.row_number().over(Window.orderBy("event_time_utc"))
)

# Define rating rules for each strategy
rating_df = df_with_id.select(
    "time_period_id",
    "event_time_utc",
    "price_eur_mwh",
    "avg_actual_total_load_mw",
    "renewable_share_pct",
    "temperature_c"
).withColumn(
    "strategy_ratings",
    F.array(
        # Strategy 1: COST_OPTIMIZER (rating based on low price)
        F.struct(
            F.lit(1).alias("strategy_id"),
            F.lit("COST_OPTIMIZER").alias("strategy_name"),
            F.when(F.col("price_eur_mwh") < 30, 5)
             .when(F.col("price_eur_mwh") < 50, 4)
             .when(F.col("price_eur_mwh") < 70, 3)
             .when(F.col("price_eur_mwh") < 100, 2)
             .otherwise(1).alias("rating")
        ),
        # Strategy 2: GREEN_PRIORITY (rating based on renewable %)
        F.struct(
            F.lit(2).alias("strategy_id"),
            F.lit("GREEN_PRIORITY").alias("strategy_name"),
            F.when(F.col("renewable_share_pct") > 0.015, 5)  # > 1.5%
             .when(F.col("renewable_share_pct") > 0.012, 4)
             .when(F.col("renewable_share_pct") > 0.010, 3)
             .when(F.col("renewable_share_pct") > 0.008, 2)
             .otherwise(1).alias("rating")
        ),
        # Strategy 3: LOAD_BALANCER (rating based on low grid load)
        F.struct(
            F.lit(3).alias("strategy_id"),
            F.lit("LOAD_BALANCER").alias("strategy_name"),
            F.when(F.col("avg_actual_total_load_mw") < 11000, 5)
             .when(F.col("avg_actual_total_load_mw") < 12000, 4)
             .when(F.col("avg_actual_total_load_mw") < 13000, 3)
             .when(F.col("avg_actual_total_load_mw") < 14000, 2)
             .otherwise(1).alias("rating")
        ),
        # Strategy 4: OPPORTUNISTIC (combined score)
        F.struct(
            F.lit(4).alias("strategy_id"),
            F.lit("OPPORTUNISTIC").alias("strategy_name"),
            (
                F.when(F.col("price_eur_mwh") < 50, 2).otherwise(0) +
                F.when(F.col("renewable_share_pct") > 0.012, 2).otherwise(0) +
                F.when(F.col("avg_actual_total_load_mw") < 12000, 1).otherwise(0)
            ).cast("int").alias("rating")
        ),
        # Strategy 5: BASELOAD (prefers moderate conditions)
        F.struct(
            F.lit(5).alias("strategy_id"),
            F.lit("BASELOAD").alias("strategy_name"),
            F.when(
                (F.col("price_eur_mwh").between(40, 80)) &
                (F.col("avg_actual_total_load_mw").between(11000, 13000)),
                5
            ).when(
                (F.col("price_eur_mwh").between(30, 100)) &
                (F.col("avg_actual_total_load_mw").between(10000, 14000)),
                3
            ).otherwise(1).alias("rating")
        )
    )
)

# Explode to create user-item-rating format
interaction_df = rating_df.select(
    "time_period_id",
    "event_time_utc",
    F.explode("strategy_ratings").alias("strategy")
).select(
    F.col("strategy.strategy_id").alias("user_id"),  # "User" = Strategy
    F.col("time_period_id").alias("item_id"),         # "Item" = Time Period
    F.col("strategy.rating").alias("rating"),
    "event_time_utc",
    F.col("strategy.strategy_name").alias("strategy_name")
).filter("rating >= 3")  # Only keep favorable ratings (3+)

print(f"\n✅ Created {interaction_df.count():,} user-item interactions (ratings ≥ 3)")
print("\nSample interactions:")
display(interaction_df.limit(10))

In [0]:
print("\n✅ STEP 2: Train ALS model")
print("="*80)

# Configure ALS model
als = ALS(
    userCol="user_id",           # Strategy ID
    itemCol="item_id",           # Time Period ID
    ratingCol="rating",          # Suitability score (1-5)
    coldStartStrategy="drop",    # Drop predictions for new users/items
    nonnegative=True,            # Force non-negative latent factors
    implicitPrefs=False,         # Using explicit ratings (1-5 scale)
    rank=10,                     # Number of latent factors
    maxIter=10,                  # Number of iterations
    regParam=0.1,                # Regularization parameter
    seed=42
)

print("\n🔧 ALS Configuration:")
print(f"  • Latent factors (rank): {als.getRank()}")
print(f"  • Max iterations: {als.getMaxIter()}")
print(f"  • Regularization: {als.getRegParam()}")
print(f"  • Cold start strategy: {als.getColdStartStrategy()}")

# Split data for validation
train_df, test_df = interaction_df.randomSplit([0.8, 0.2], seed=42)
print(f"\n📊 Data split:")
print(f"  • Training interactions: {train_df.count():,}")
print(f"  • Test interactions: {test_df.count():,}")

# Train model
import time
start_time = time.time()
als_model = als.fit(train_df)
training_time = time.time() - start_time

print(f"\n✅ Model training completed in {training_time:.2f} seconds")
print(f"\n🎯 Model learned:")
print(f"  • {als_model.userFactors.count()} strategy profiles (user factors)")
print(f"  • {als_model.itemFactors.count():,} time period profiles (item factors)")

In [0]:
print("\n✅ STEP 3: Generate Top-5 recommendations for all strategies")
print("="*80)

# Generate recommendations for all strategies
user_recs = als_model.recommendForAllUsers(5)

print(f"\n📋 Generated recommendations for {user_recs.count()} strategies")
print("\nSample recommendations (strategy → top 5 time periods):")
display(user_recs.limit(3))

In [0]:
print("\n✅ STEP 4: Explode and enrich recommendations with time details")
print("="*80)

# Explode recommendations and join with original data
recs_exploded = user_recs.select(
    F.col("user_id").alias("strategy_id"),
    F.posexplode("recommendations").alias("rank", "recommendation")
).select(
    "strategy_id",
    (F.col("rank") + 1).alias("recommendation_rank"),
    F.col("recommendation.item_id").alias("time_period_id"),
    F.col("recommendation.rating").alias("predicted_score")
)

# Join with original data to get time and conditions
recs_enriched = recs_exploded.join(
    df_with_id.select(
        "time_period_id",
        "event_time_utc",
        "price_eur_mwh",
        "avg_actual_total_load_mw",
        "renewable_share_pct",
        "temperature_c",
        "wind_speed_ms"
    ),
    "time_period_id",
    "left"
)

# Add strategy names
strategy_mapping = spark.createDataFrame([
    (1, "COST_OPTIMIZER"),
    (2, "GREEN_PRIORITY"),
    (3, "LOAD_BALANCER"),
    (4, "OPPORTUNISTIC"),
    (5, "BASELOAD")
], ["strategy_id", "strategy_name"])

recs_final = recs_enriched.join(strategy_mapping, "strategy_id").select(
    "strategy_id",
    "strategy_name",
    "recommendation_rank",
    "time_period_id",
    "event_time_utc",
    F.round("predicted_score", 2).alias("predicted_score"),
    F.round("price_eur_mwh", 2).alias("price_eur_mwh"),
    F.round("avg_actual_total_load_mw", 0).alias("load_mw"),
    F.round("renewable_share_pct", 4).alias("renewable_share_pct"),
    F.round("temperature_c", 1).alias("temperature_c"),
    F.round("wind_speed_ms", 1).alias("wind_speed_ms")
).orderBy("strategy_id", "recommendation_rank")

print(f"\n✅ Enriched {recs_final.count()} recommendations with energy data")
print("\n🎯 Top-5 Recommendations per Strategy:")
print("="*80)
display(recs_final)

In [0]:
print("\n✅ STEP 5: Analyze recommendation patterns")
print("="*80)

# Analyze average conditions in top recommendations
strategy_analysis = recs_final.groupBy("strategy_name").agg(
    F.round(F.avg("predicted_score"), 2).alias("avg_predicted_score"),
    F.round(F.avg("price_eur_mwh"), 2).alias("avg_price_eur_mwh"),
    F.round(F.avg("load_mw"), 0).alias("avg_load_mw"),
    F.round(F.avg("renewable_share_pct"), 4).alias("avg_renewable_share_pct"),
    F.round(F.min("price_eur_mwh"), 2).alias("min_price"),
    F.round(F.max("renewable_share_pct"), 4).alias("max_renewable")
).orderBy("strategy_name")

print("\n📊 Average Conditions in Top-5 Recommendations per Strategy:")
display(strategy_analysis)

# Show specific examples for each strategy
print("\n🔍 Example: Top Recommendation for Each Strategy")
top_1_per_strategy = recs_final.filter("recommendation_rank = 1").select(
    "strategy_name",
    "event_time_utc",
    "predicted_score",
    "price_eur_mwh",
    "renewable_share_pct",
    "load_mw"
)
display(top_1_per_strategy)

In [0]:
print("\n✅ STEP 6: Evaluate model performance")
print("="*80)

# Generate predictions on test set
test_predictions = als_model.transform(test_df)

# Calculate RMSE
from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(test_predictions.filter("prediction IS NOT NULL"))

print(f"\n📊 Model Performance:")
print(f"  • Test RMSE: {rmse:.4f}")
print(f"  • Rating scale: 1-5")
print(f"  • Interpretation: Average prediction error of {rmse:.2f} rating points")

# Show sample predictions vs actuals
print("\n🔍 Sample predictions vs actual ratings:")
sample_preds = test_predictions.filter("prediction IS NOT NULL").select(
    "user_id",
    "item_id",
    F.round("rating", 1).alias("actual_rating"),
    F.round("prediction", 2).alias("predicted_rating"),
    F.round(F.abs(F.col("rating") - F.col("prediction")), 2).alias("error")
).limit(10)

display(sample_preds)

In [0]:
print("\n✅ STEP 7: Save recommendations to production table")
print("="*80)

# Add metadata
from datetime import datetime
recs_with_metadata = recs_final.withColumn(
    "model_version",
    F.lit("als_v1_rank10")
).withColumn(
    "generated_at",
    F.lit(datetime.now())
).withColumn(
    "model_rmse",
    F.lit(round(rmse, 4))
)

# Save to Delta table
output_table = f"{CATALOG}.{SCHEMA}.h2_gold_als_recommendations"

recs_with_metadata.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

print(f"\n✅ Saved {recs_with_metadata.count()} recommendations to: {output_table}")
print("\n📋 Table Schema:")
spark.table(output_table).printSchema()

# Verify save
verify_count = spark.table(output_table).count()
print(f"\n✓ Verification: {verify_count:,} records in production table")

## ✅ Day 9 Complete: Recommendation System with ALS

### What We Built

**Collaborative Filtering System for Energy Data**:
* **User-Item Framework**: 5 operating strategies × 17,535 time periods
* **Rating System**: 1-5 suitability scores based on energy conditions
* **ALS Model**: 10 latent factors, trained on favorable periods (rating ≥ 3)
* **Output**: Top-5 time period recommendations per strategy

### Key Concepts Demonstrated

1. **Collaborative Filtering**: Learned patterns from strategy-time period interactions
2. **Matrix Factorization**: Decomposed sparse rating matrix into dense factor matrices
3. **Cold Start Handling**: Used `coldStartStrategy="drop"` to handle missing data
4. **Implicit vs Explicit**: Used explicit ratings (1-5 scale) based on conditions

### Honest Limitations

| Limitation | Impact |
| --- | --- |
| Only 5 "users" (strategies) | Matrix too small for meaningful latent factor discovery |
| Ratings are heuristic, not real | ALS learns patterns we already hardcoded |
| No real collaborative signal | A SQL query with `WHERE`/`ORDER BY` produces equivalent results |
| No cold-start challenge | All items have known features — no missing data to predict |

### When ALS Becomes Valuable (Future Scope)

| Expansion | How ALS Helps |
| --- | --- |
| **Multi-country** (NL, DE, FR, BE) | Countries as "users" — discover cross-border patterns |
| **Real operators** (50+ plant operators) | Genuine diverse preferences → real collaborative filtering |
| **Production logs** (implicit feedback) | Did they actually run the electrolyzer? Real signal |
| **14 generation types** (not consolidated) | Richer item features for hybrid content + collaborative filtering |

### Better Alternatives for Current Scope

* **Simple SQL optimization**: `WHERE price < 30 AND renewable_share > X ORDER BY price`
* **Multi-objective optimization**: Linear programming for production scheduling
* **Clustering**: Group similar time periods (k-means on energy features)
* **Richer regression**: Use all 14 generation types as features in RF/GBT

### Production Assets Created

* **Table**: `h2_gold_als_recommendations` (25 records = 5 strategies × 5 recommendations)
* **Model**: ALS with 10 latent factors (user + item factors stored in model)
* **Metadata**: Model version, RMSE, generation timestamp

### Challenge Compliance

✅ Create rating mapping → 5-point scale based on energy conditions  
✅ Train ALS model → 10 latent factors, RMSE evaluation  
✅ Generate Top-5 recommendations → Per-strategy recommendations with enriched data

---

**🎯 Day 9 Achievement Unlocked**: Demonstrated ALS collaborative filtering technique. For production use, recommend investing in multi-objective optimization or richer supervised models instead.

**📚 Official Challenge Reference**: https://spark.apache.org/docs/latest/ml-collaborative-filtering.html